# Task 13 -- Diurnal wave-signal monitoring trigger (`docs/open_questions.md` §7.2)

This is a **re-runnable trigger check**, not a one-time hypothesis test --
`open_questions.md` §7.2 is explicitly "not an active concern... only if
this pipeline is ever pointed at a higher-resolution regional wave model."
See `tasks/13-diurnal-signal-monitoring-trigger.md` for the full
narrative-check steps (confirming the wave source is still GFS-Wave-class
global resolution, confirming `harmonic_climatology.md`'s diurnal-variable
list still excludes wave fields). This notebook runs the one part of that
check that's real, runnable code: a standing real-data baseline for `wsh`'s
diurnal-term significance, so a future re-run has something concrete to
diff against if the trigger ever fires.

In [ ]:
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import tensorly as tl
import xarray as xr
from tensorly.decomposition import partial_tucker

from train.ingest import load_training_window

tl.set_backend("numpy")

SEED = 42
rng = np.random.default_rng(SEED)
DATA_DIR = Path("../data")

# Window entirely inside era 3 (starts 2025-10-27, core.archive_eras.ARCHIVE_ERAS);
# load_training_window rejects any window spanning more than one era.
T_START = datetime(2025, 11, 1, tzinfo=timezone.utc)
T_END = datetime(2026, 2, 1, tzinfo=timezone.utc)

VARIABLE = "wsh"
X_raw = load_training_window(DATA_DIR, T_START, T_END, [VARIABLE])


## Baseline: fit a diurnal term for `wsh` (normally excluded) and measure its size

In [ ]:
from train.climatology import ClimatologyRemover

remover_diurnal = ClimatologyRemover(diurnal_variables=[VARIABLE]).fit(X_raw)
coeffs = remover_diurnal.coeffs_[VARIABLE]  # (n_terms, lat, lon); diurnal cos/sin are the last 2 rows
diurnal_amplitude = np.hypot(coeffs[-2], coeffs[-1])  # per-cell diurnal harmonic amplitude

remover_default = ClimatologyRemover().fit(X_raw)  # wsh excluded by default -- no diurnal term to compare
seasonal_amplitude = np.hypot(remover_diurnal.coeffs_[VARIABLE][2], remover_diurnal.coeffs_[VARIABLE][3])  # annual

ocean_mask = ~np.isnan(diurnal_amplitude)
ratio = np.nanmedian(diurnal_amplitude[ocean_mask]) / np.nanmedian(seasonal_amplitude[ocean_mask])
print(f"median diurnal amplitude: {np.nanmedian(diurnal_amplitude[ocean_mask]):.5f} m")
print(f"median annual amplitude: {np.nanmedian(seasonal_amplitude[ocean_mask]):.5f} m")
print(f"diurnal/annual ratio: {ratio:.4f} (baseline -- expected small; re-run and compare if the trigger fires)")


## Trigger-check narrative (manual, not code)

1. Re-read `docs/noaa_etl/pipeline.md`/`noaa_ingestion.md` -- confirm the
   documented wave source is still GFS-Wave at global (0.25°) resolution.
2. Confirm `harmonic_climatology.md`'s default diurnal-variable list still
   excludes all wave-grid variables.
3. If both are unchanged, this baseline ratio is the record -- **no action
   needed**, but the check should be re-run periodically, not treated as
   permanently closed.
4. If either has changed (higher-resolution regional model, or a
   locally-wind-driven short-fetch application), re-run the cell above and
   compare the new diurnal/annual ratio against this baseline; a material
   increase is the trigger to actually revisit `open_questions.md` §7.2.

Record the check date and outcome in `tasks/13-diurnal-signal-monitoring-trigger.md` §4 --
note explicitly that a "no change" conclusion **expires** and should be
re-verified, unlike the other 14 tasks.